# DTL-5G Code Sample 1 — Source-Domain BiLSTM Pretraining
## Objective 1, iteration 1 (`random_state = 42`)

This notebook trains the source-domain BiLSTM for one Objective-1 iteration.

Five Objective-1 iterations use fixed random states:
`42, 142, 12, 4, 80`.

### Configuration
- source traffic from both network slices
- eight DDoS attack categories
- eight selected flow features
- each attack category downsampled to approximately one-eighth of benign traffic
- 80/20 stratified train/test split
- StandardScaler fitted on training data and applied to test data
- source test set reduced to 6,085 samples
- BiLSTM with 9 Keras layers and 99,970 parameters
- L2 regularization = 0.001
- dropout = 0.5
- Adam optimizer with learning rate = 1e-5
- up to 200 epochs with early stopping
- attack is the positive class for recall, precision, and F1


## 1. Imports and configuration

In [ ]:
from pathlib import Path
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Activation, Bidirectional, Dense, Dropout, LSTM
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
)

OBJECTIVE1_RANDOM_STATES = [42, 142, 12, 4, 80]
ITERATION_SEED = OBJECTIVE1_RANDOM_STATES[0]  # Objective 1, iteration 1

DATA_ROOT = Path(os.getenv("DTL5G_SOURCE_DATA_ROOT", "./data/VM")).expanduser()
ATTACK_DIR = DATA_ROOT / "Attack" / "12 May"
BENIGN_DIR = DATA_ROOT / "Benign"
TARGET_TEST_SIZE = 6_085
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

In [ ]:
# Reproducible model initialization, dropout masks, NumPy operations,
# and TensorFlow random operations for this iteration.
keras.utils.set_random_seed(ITERATION_SEED)

# Request deterministic TensorFlow kernels where supported.
try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print(f"Deterministic TensorFlow operations unavailable: {exc}")

## 2. Features and source-data manifest

In [ ]:
FEATURES = ['Flow Duration', 'Fwd Pkt Len Std', 'ACK Flag Cnt', 'Protocol', 'Tot Fwd Pkts', 'TotLen Fwd Pkts', 'TotLen Bwd Pkts', 'Tot Bwd Pkts']
LABEL_COLUMN = "Label"

ATTACK_FILES = {'UDP_Flood': ['12udpfloodupf2.pcap_Flow.csv', '12udpfloodupf1.pcap_Flow.csv'], 'TCP_YMAS': ['12tcpymasupf2.pcap_Flow.csv', '12tcpymasupf1.pcap_Flow.csv'], 'TCP_XMAS': ['12tcpxmasupf2.pcap_Flow.csv', '12tcpxmasupf1.pcap_Flow.csv'], 'TCP_URG': ['12tcpurgupf2.pcap_Flow.csv', '12tcpurgupf1.pcap_Flow.csv'], 'TCP_SYN': ['12tcpsynupf2.pcap_Flow.csv', '12tcpsynupf1.pcap_Flow.csv'], 'TCP_PUSH': ['12tcppushupf2.pcap_Flow.csv', '12tcppushupf1.pcap_Flow.csv'], 'TCP_FIN': ['12tcpfinupf2.pcap_Flow.csv', '12tcpfinupf1.pcap_Flow.csv'], 'TCP_ACK': ['12tcpackupf2.pcap_Flow.csv', '12tcpackupf1.pcap_Flow.csv']}
BENIGN_FILES = ['30upf2.pcap_Flow.csv', '30upf1.pcap_Flow.csv', '30pingupf2.pcap_Flow.csv', '30pingupf1.pcap_Flow.csv', '30Aprilupf2.pcap_Flow.csv', '30Aprilupf1.pcap_Flow.csv', '30-1upf2.pcap_Flow.csv', '30-1upf1.pcap_Flow.csv', '29Aprilupf2.pcap_Flow.csv', '29Aprilupf1.pcap_Flow.csv', '2mayupf2.pcap_Flow.csv', '2mayupf1.pcap_Flow.csv', '2maysshupf2.pcap_Flow.csv', '2maysshupf1.pcap_Flow.csv', '2maydownloadupf2.pcap_Flow.csv', '2maydownloadupf1.pcap_Flow.csv', '2maydownload-1upf2.pcap_Flow.csv', '2maydownload-1upf1.pcap_Flow.csv']

def assert_unique_manifest(name, filenames):
    duplicates = sorted({item for item in filenames if filenames.count(item) > 1})
    if duplicates:
        raise ValueError(f"{name} contains duplicate filenames: {duplicates}")

assert_unique_manifest("BENIGN_FILES", BENIGN_FILES)
assert_unique_manifest(
    "ATTACK_FILES",
    [filename for filenames in ATTACK_FILES.values() for filename in filenames],
)


## 3. Load the combined two-slice source domain

In [ ]:
def load_group(directory, filenames, label):
    paths = [directory / name for name in filenames]
    missing = [str(path) for path in paths if not path.exists()]

    if missing:
        raise FileNotFoundError(
            "Missing source CSV files. Set DTL5G_SOURCE_DATA_ROOT correctly.\n"
            + "\n".join(missing[:10])
        )

    frame = pd.concat((pd.read_csv(path) for path in paths), ignore_index=True)
    frame[LABEL_COLUMN] = label
    return frame[FEATURES + [LABEL_COLUMN]]


benign_df = load_group(BENIGN_DIR, BENIGN_FILES, "Benign")
attack_frames = {
    attack_name: load_group(ATTACK_DIR, filenames, "attack")
    for attack_name, filenames in ATTACK_FILES.items()
}

print(f"{'Benign':>10}: {len(benign_df):,}")
for attack_name, frame in attack_frames.items():
    print(f"{attack_name:>10}: {len(frame):,}")


## 4. Cleaning and Objective-1 balancing

In [ ]:
benign_df = benign_df.replace([np.inf, -np.inf], np.nan)
attack_frames = {
    name: frame.replace([np.inf, -np.inf], np.nan)
    for name, frame in attack_frames.items()
}

samples_per_attack = len(benign_df) // 8

sampled_attacks = {
    name: frame.sample(
        n=samples_per_attack,
        random_state=ITERATION_SEED,
        replace=False,
    )
    for name, frame in attack_frames.items()
}

print("Benign:", f"{len(benign_df):,}")
print("Per attack category:", f"{samples_per_attack:,}")

def validate_selected_features(frame, name):
    values = frame[FEATURES].to_numpy(dtype=np.float64)
    invalid = ~np.isfinite(values)

    if invalid.any():
        rows, cols = np.where(invalid)
        examples = [
            (int(r), FEATURES[int(c)])
            for r, c in zip(rows[:10], cols[:10])
        ]
        raise ValueError(
            f"{name} contains {invalid.sum()} non-finite selected-feature values. "
            f"Examples: {examples}"
        )

    print(f"{name}: selected features are finite.")

validate_selected_features(benign_df, "Benign source traffic")
for attack_name, attack_frame in attack_frames.items():
    validate_selected_features(attack_frame, attack_name)


## 5. Feature matrix and binary labels

In [ ]:
dataset = pd.concat([benign_df, *sampled_attacks.values()], ignore_index=True)

X = dataset[FEATURES].to_numpy(dtype=np.float32)
y = dataset[LABEL_COLUMN].map({"attack": 0, "Benign": 1}).to_numpy(dtype=np.int32)

print("X:", X.shape)
print("y:", y.shape)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))

## 6. Stratified 80/20 split and StandardScaler

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=ITERATION_SEED,
    stratify=y,
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = np.expand_dims(X_train, axis=-1)
X_test = np.expand_dims(X_test, axis=-1)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

## 7. Match the source test size to the 5G-NIDD target test size

In [ ]:
rng = np.random.default_rng(ITERATION_SEED)
selected_indices = rng.choice(
    len(X_test),
    size=TARGET_TEST_SIZE,
    replace=False,
)
X_test = X_test[selected_indices]
y_test = y_test[selected_indices]

assert len(X_test) == 6_085
print("Reduced source test set:", X_test.shape)

## 8. BiLSTM architecture

In [ ]:
def build_bilstm(input_shape=(8, 1), num_classes=2):
    """
    BiLSTM graph satisfying the explicit experiment specifications:
      - 9 top-level Keras layers
      - 99,970 parameters
      - L2 coefficient 0.001
      - dropout rate 0.5
      - Adam learning rate 1e-5

    Dense(256) + explicit ReLU is mathematically equivalent to
    Dense(256, activation="relu") but keeps the reported layer count explicit.
    """
    reg = regularizers.l2(0.001)

    inputs = keras.Input(shape=input_shape, name="input")
    x = Bidirectional(
        LSTM(
            64,
            activation="tanh",
            kernel_regularizer=reg,
        ),
        name="bilstm",
    )(inputs)
    x = Dropout(0.5, name="dropout_1")(x)
    x = Dense(
        256,
        activation=None,
        kernel_regularizer=reg,
        name="dense_256",
    )(x)
    x = Activation("relu", name="relu_256")(x)
    x = Dropout(0.5, name="dropout_2")(x)
    x = Dense(
        128,
        activation="relu",
        kernel_regularizer=reg,
        name="dense_128",
    )(x)
    x = Dropout(0.5, name="dropout_3")(x)
    outputs = Dense(
        num_classes,
        activation="sigmoid",
        kernel_regularizer=reg,
        name="output",
    )(x)

    model = keras.Model(inputs=inputs, outputs=outputs, name="DTL5G_BiLSTM")
    model.compile(
        loss="binary_crossentropy",
        optimizer=keras.optimizers.Adam(learning_rate=1e-5),
        metrics=["accuracy"],
    )

    assert len(model.layers) == 9, f"Expected 9 layers, found {len(model.layers)}"
    assert model.count_params() == 99_970, (
        f"Expected 99,970 parameters, found {model.count_params():,}"
    )
    return model

model = build_bilstm()
model.summary()

print("Top-level layers:", len(model.layers))
print("Total parameters:", f"{model.count_params():,}")

### Parameter audit
- Bidirectional LSTM(64): 33,792
- Dense(256): 33,024
- Dense(128): 32,896
- Dense(2): 258
- Input, Dropout and Activation layers: 0 parameters

**Total = 99,970 parameters.**

## 9. Train

In [ ]:
y_train_one_hot = keras.utils.to_categorical(y_train.astype(float), num_classes=2)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=15,
    min_delta=0.0001,
)

start = time.perf_counter()
history = model.fit(
    X_train,
    y_train_one_hot,
    epochs=200,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1,
)
training_seconds = time.perf_counter() - start

print(f"Training time: {training_seconds / 60:.2f} min")
print(f"Training time/sample: {(training_seconds * 1000) / len(X_train):.4f} ms")

## 10. Save the iteration-1 source checkpoint

In [ ]:
SOURCE_MODEL_PATH = MODEL_DIR / "BiLSTM_VM_Obj1_1.keras"
model.save(SOURCE_MODEL_PATH)
print("Saved:", SOURCE_MODEL_PATH)

# Notebook 3 uses this checkpoint by default.
# To use a checkpoint from another source run, set DTL5G_SOURCE_MODEL.


## 11. Evaluate

In [ ]:
def evaluate_attack_positive(model, X_test, y_test):
    """Evaluate using the evaluation convention: attack is the positive class."""
    probabilities = model.predict(X_test, verbose=0)
    predictions = probabilities.argmax(axis=1)

    metrics_out = {
        "accuracy": accuracy_score(y_test, predictions),
        "recall": recall_score(y_test, predictions, pos_label=0, zero_division=0),
        "precision": precision_score(y_test, predictions, pos_label=0, zero_division=0),
        "f1": f1_score(y_test, predictions, pos_label=0, zero_division=0),
        "auc": roc_auc_score((y_test == 0).astype(int), probabilities[:, 0]),
    }

    for key, value in metrics_out.items():
        print(f"{key.capitalize():>9}: {value:.4f}")

    return probabilities, predictions, metrics_out

probabilities, predictions, result = evaluate_attack_positive(
    model, X_test, y_test
)

## 12. Inference timing

In [ ]:
def benchmark(device):
    with tf.device(device):
        start = time.perf_counter()
        _ = model.predict(X_test, verbose=0)
        elapsed = time.perf_counter() - start
    return elapsed, elapsed * 1000 / len(X_test)

cpu_total, cpu_each = benchmark("/CPU:0")
print(f"CPU: {cpu_each:.4f} ms/sample")

if tf.config.list_physical_devices("GPU"):
    gpu_total, gpu_each = benchmark("/GPU:0")
    print(f"GPU: {gpu_each:.4f} ms/sample")

## 13. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
cm_df = pd.DataFrame(
    cm,
    index=["Attack", "Benign"],
    columns=["Attack", "Benign"],
)

plt.figure(figsize=(6, 4))
sns.heatmap(cm_df, annot=True, fmt="d", cbar=False)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("Source BiLSTM — Objective 1, iteration 1")
plt.tight_layout()
plt.show()

## Five-run aggregation

This notebook contains one Objective-1 iteration. The complete experiment uses
the fixed seeds `42, 142, 12, 4, 80`, and experiment-level metrics are obtained
by averaging the corresponding results across the five runs.
